In [2]:
import os
from pathlib import Path

parent_dir = Path().cwd().parent
os.chdir(parent_dir)

In [3]:
import pandas as pd

from src.data_pipeline.loader import DataPreparer
from src.data_pipeline.pipeline_models import data_pipeline_settings

In [4]:
loader = DataPreparer(data_pipeline_settings)

df = loader.load_data()

In [5]:
df

,Unnamed: 0.1,Unnamed: 0,date,T1,P2,F3,P4,F5,T6_x,F7,...,F17,T18_y,F19_y,Q20,Q21,F22,T23,P24,F25_y,F26_y
0,0,0,2023-01-01 00:00:00,130.372528,3.540594,68.645889,3.848177,103.866081,234.733780,217.544907,...,208.366318,66.580772,167.797852,7143.850098,7.699553,10507.554688,234.196854,0.594864,13242.359375,201.230698
1,1,1,2023-01-01 00:10:00,130.312668,3.535826,68.768883,3.847023,103.823425,234.711334,217.332169,...,206.546371,66.387772,168.481689,7596.144531,6.959560,10526.750000,233.974182,0.595058,13214.491211,201.488571
2,2,2,2023-01-01 00:20:00,130.254074,3.532611,68.994743,3.844830,98.811737,234.677292,217.119431,...,204.726425,66.762436,167.650955,7656.547363,6.510911,11271.607422,233.929260,0.592871,13313.846680,201.574692
3,3,3,2023-01-01 00:30:00,130.195465,3.532775,68.718155,3.841944,97.248512,234.480499,216.560745,...,205.050873,67.137100,166.420822,7939.474121,5.955529,12260.086914,234.033798,0.595605,13290.474609,201.460220
4,4,4,2023-01-01 00:40:00,130.131912,3.532940,68.386543,3.839058,103.873940,234.283707,215.707336,...,206.608307,67.722862,166.531998,8020.019531,5.592346,13249.763672,233.899139,0.595312,13065.634766,203.041473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189212,189212,189212,2026-08-06 23:20:00,131.194336,3.836440,107.416397,4.161318,439.080719,229.764740,291.509308,...,304.541504,65.573502,238.619781,7662.971680,7.985026,8585.574219,232.503662,0.579832,13531.119141,296.155029
189213,189213,189213,2026-08-06 23:30:00,131.336182,3.837928,107.527336,4.162167,425.742493,229.890030,291.837250,...,303.739502,65.747917,240.845184,7640.085938,7.559968,8608.854492,232.568710,0.576999,13404.511719,297.104767
189214,189214,189214,2026-08-06 23:40:00,131.114349,3.839417,107.628471,4.163015,438.722351,230.018845,292.066010,...,301.167603,65.970657,244.287338,7656.569824,7.293701,8598.671875,232.635071,0.574730,13146.961914,297.935822
189215,189215,189215,2026-08-06 23:50:00,131.180481,3.837819,107.398705,4.160222,425.805847,230.147659,292.128174,...,300.048645,65.992149,252.587006,7431.098633,7.429495,8613.925781,232.707993,0.573450,13283.888672,298.311127


In [6]:
print((df['Unnamed: 0.1'] == df['Unnamed: 0']).all())
print((df['Unnamed: 0.1'] == df.index.to_series()).all())

True
True


In [7]:
# Исключаем период простоя
downtime_start = pd.Timestamp("2026-06-18 06:00:00")
downtime_end = pd.Timestamp("2026-07-02 14:00:00")

df_hist = df[
    ~df["date"].between(
        downtime_start,
        downtime_end,
        inclusive="both",
    )
].copy()


# Считаем исторические min / max / median
# только для числовых тегов
numeric_columns = df_hist.select_dtypes(include="number").columns

# Служебные индексы не являются измерениями
numeric_columns = numeric_columns.drop(
    ["Unnamed: 0", "Unnamed: 0.1"],
    errors="ignore",
)

historical_stats = (
    df_hist[numeric_columns]
    .agg(["min", "max", "median"])
    .T
    .reset_index()
    .rename(columns={"index": "tag"})
)


# Сохраняем результат
output_path = (
    data_pipeline_settings.converted_data_path
    / "historical_stats.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

historical_stats.to_csv(
    output_path,
    index=False,
)

print(f"Saved: {output_path}")
historical_stats

Saved: data/converted/historical_stats.csv


,tag,min,max,median
0,T1,0.012207,353.835297,127.034908
1,P2,-0.264300,307.000000,3.511778
2,F3,0.616474,307.000000,79.746258
3,P4,-0.264293,307.000000,3.802851
4,F5,-169.239685,6211.660156,-13.730229
...,...,...,...,...
93,F22,0.000000,21173.238281,11013.724121
94,T23,-1.559331,307.000000,235.281158
95,P24,-0.007612,307.000000,0.584961
96,F25_y,0.000000,18234.570312,13117.722168
